# Keithley 2460 guarded bipolar resistor sweep

This notebook reproduces the six-point linearity/repeatability follow-up for a **nominal 100 Ω resistor connected to the front terminals with four-wire sense**. Never run it with a coin cell, battery, capacitor, unknown load, or another controller attached.

Confirm the OUTPUT indicator is physically off, then set `CONFIRM_FRONT_4WIRE_100_OHM` to `True`. The sweep uses −1 mA to +1 mA, a fixed 1 mA source range, five readings per point, and the instrument's minimum 0.2 V current-source voltage limit. Output is switched OFF between points and cleanup programs 0 A. If communication fails while output might be on, use the front-panel OUTPUT control immediately.

The resistor is uncalibrated, so this is not an accuracy or hardware-acceptance test.

In [ ]:
import json
import math
import statistics
from datetime import UTC, datetime

from ophyd_electrochemistry.keithley.k2460.transport import (
    PyVisaKeithley2460Transport,
    VisaTransportConfig,
)

RESOURCE = "TCPIP0::169.254.77.125::5025::SOCKET"
CONFIRM_FRONT_4WIRE_100_OHM = False  # Change only after the physical checklist.
CURRENTS_A = (-1e-3, -5e-4, -1e-4, 1e-4, 5e-4, 1e-3)
READINGS_PER_POINT = 5
VOLTAGE_LIMIT_V = 0.2


def finite_number(reply: str, label: str) -> float:
    try:
        value = float(reply.strip())
    except ValueError as exc:
        raise RuntimeError(f"{label}: nonnumeric reply {reply!r}") from exc
    if not math.isfinite(value):
        raise RuntimeError(f"{label}: nonfinite reply {reply!r}")
    return value


def require_token(actual: str, allowed: tuple[str, ...], label: str) -> None:
    if actual not in allowed:
        raise RuntimeError(f"{label}: expected one of {allowed}, received {actual!r}")


def read_pair(instrument: PyVisaKeithley2460Transport) -> tuple[float, float]:
    raw = instrument.query(':READ? "defbuffer1", SOURce, READing')
    fields = [field.strip() for field in raw.split(",")]
    if len(fields) != 2:
        raise RuntimeError(f"expected source/read pair, received {raw!r}")
    return finite_number(fields[0], "source readback"), finite_number(fields[1], "voltage reading")

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM:
    raise RuntimeError(
        "Physical setup not confirmed. Complete the checklist and set "
        "CONFIRM_FRONT_4WIRE_100_OHM = True."
    )

config = VisaTransportConfig(
    resource_name=RESOURCE,
    backend="@py",
    required_language="SCPI",
    io_timeout_ms=5_000,
)
report = {
    "schema": "k2460-resistor-bipolar-sweep-v1",
    "observed_at_utc": datetime.now(UTC).isoformat(),
    "resource_name": RESOURCE,
    "nominal_resistance_ohm": 100.0,
    "load_calibrated": False,
    "requested_currents_a": list(CURRENTS_A),
    "readings_per_point": READINGS_PER_POINT,
    "voltage_limit_v": VOLTAGE_LIMIT_V,
    "points": [],
}
instrument = PyVisaKeithley2460Transport(config)
cleanup_error = None

try:
    instrument.connect()
    report["identity"] = instrument.identity.raw
    require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "preflight output")
    require_token(instrument.query(":ROUTe:TERMinals?"), ("FRON", "FRONT"), "terminals")

    for command in (
        ":FORMat:DATA ASCii",
        ":SOURce:FUNCtion CURRent",
        ":SOURce:CURRent 0",
        ":SOURce:CURRent:RANGe 0.001",
        ":SOURce:CURRent:VLIMit 0.2",
        ":SOURce:CURRent:READ:BACK ON",
        ':SENSe:FUNCtion "VOLTage"',
        ":SENSe:VOLTage:RSENse ON",
        ":SENSe:VOLTage:RANGe 0.2",
        ":SENSe:VOLTage:NPLCycles 1",
        ":SENSe:VOLTage:AVERage OFF",
        ":SENSe:VOLTage:RELative:STATe OFF",
        ":SENSe:COUNt 1",
    ):
        instrument.write(command)

    require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "configured output")
    require_token(instrument.query(":ROUTe:TERMinals?"), ("FRON", "FRONT"), "configured terminals")
    require_token(instrument.query(":SOURce:FUNCtion?"), ("CURR", "CURRENT"), "source function")
    require_token(instrument.query(":SENSe:VOLTage:RSENse?"), ("1", "ON"), "remote sense")
    if (
        abs(finite_number(instrument.query(":SOURce:CURRent:RANGe?"), "source range") - 0.001)
        > 1e-12
    ):
        raise RuntimeError("source range verification failed")
    if (
        abs(finite_number(instrument.query(":SOURce:CURRent:VLIMit?"), "voltage limit") - 0.2)
        > 1e-9
    ):
        raise RuntimeError("voltage limit verification failed")

    all_samples = []
    for requested_current_a in CURRENTS_A:
        instrument.write(f":SOURce:CURRent {requested_current_a:.12g}")
        require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "pre-point output")
        programmed_a = finite_number(instrument.query(":SOURce:CURRent?"), "programmed current")
        if abs(programmed_a - requested_current_a) > max(1e-12, abs(requested_current_a) * 1e-6):
            raise RuntimeError("programmed current verification failed")

        samples = []
        try:
            instrument.write(":OUTPut:STATe ON")
            require_token(instrument.query(":OUTPut:STATe?"), ("1", "ON"), "energized output")
            for _ in range(READINGS_PER_POINT):
                measured_current_a, measured_voltage_v = read_pair(instrument)
                sample = {
                    "measured_current_a": measured_current_a,
                    "measured_voltage_v": measured_voltage_v,
                    "resistance_ohm": measured_voltage_v / measured_current_a,
                }
                samples.append(sample)
                all_samples.append(sample)
        finally:
            instrument.write(":OUTPut:STATe OFF")
            require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "post-point output")

        limit_tripped = instrument.query(":SOURce:CURRent:VLIMit:TRIPped?")
        currents = [sample["measured_current_a"] for sample in samples]
        voltages = [sample["measured_voltage_v"] for sample in samples]
        resistances = [sample["resistance_ohm"] for sample in samples]
        point = {
            "requested_current_a": requested_current_a,
            "mean_current_a": statistics.fmean(currents),
            "stdev_current_a": statistics.stdev(currents),
            "mean_voltage_v": statistics.fmean(voltages),
            "stdev_voltage_v": statistics.stdev(voltages),
            "mean_resistance_ohm": statistics.fmean(resistances),
            "stdev_resistance_ohm": statistics.stdev(resistances),
            "voltage_limit_tripped": limit_tripped == "1",
            "output_off_confirmed": True,
            "samples": samples,
        }
        report["points"].append(point)
        if limit_tripped != "0" or not (80 <= point["mean_resistance_ohm"] <= 120):
            raise RuntimeError(f"point plausibility failed: {point}")
        if point["mean_voltage_v"] * requested_current_a <= 0:
            raise RuntimeError(f"polarity failed: {point}")
        instrument.write(":SOURce:CURRent 0")

    x = [sample["measured_current_a"] for sample in all_samples]
    y = [sample["measured_voltage_v"] for sample in all_samples]
    x_mean, y_mean = statistics.fmean(x), statistics.fmean(y)
    sxx = sum((value - x_mean) ** 2 for value in x)
    slope_ohm = (
        sum((current - x_mean) * (voltage - y_mean) for current, voltage in zip(x, y, strict=True))
        / sxx
    )
    intercept_v = y_mean - slope_ohm * x_mean
    residuals = [
        voltage - (slope_ohm * current + intercept_v) for current, voltage in zip(x, y, strict=True)
    ]
    ss_res = sum(value**2 for value in residuals)
    ss_tot = sum((value - y_mean) ** 2 for value in y)
    report["linear_fit"] = {
        "slope_ohm": slope_ohm,
        "intercept_v": intercept_v,
        "r_squared": 1 - ss_res / ss_tot,
        "max_abs_residual_v": max(abs(value) for value in residuals),
        "sample_count": len(all_samples),
    }
    if not (80 <= slope_ohm <= 120) or abs(intercept_v) > 5e-3:
        raise RuntimeError(f"fit plausibility failed: {report['linear_fit']}")
finally:
    if instrument.connected:
        try:
            instrument.write(":OUTPut:STATe OFF")
            require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "final output")
            instrument.write(":SOURce:CURRent 0")
            report["final_output_enabled"] = False
            report["final_programmed_current_a"] = finite_number(
                instrument.query(":SOURce:CURRent?"), "final programmed current"
            )
        except BaseException as exc:
            cleanup_error = exc
    try:
        instrument.close()
    except BaseException as exc:
        cleanup_error = cleanup_error or exc
    if cleanup_error is not None:
        raise RuntimeError(
            f"cleanup failed; inspect front panel: {cleanup_error}"
        ) from cleanup_error

print(json.dumps(report, indent=2))

## Observed result on 2026-10-05

The 30 readings produced a fitted resistance of **102.502310 Ω**, intercept **−33.020 µV**, R² **0.999999999914**, and worst absolute residual **1.252 µV**. Every point had the correct polarity, no voltage-limit trip occurred, and an independent session confirmed output OFF and programmed current 0 A.

Full samples are retained in `docs/evidence/k2460-2026-10-05-resistor-bipolar-sweep.json`.